# Testando hipóteses: seca e o salto de focos em 2017

Duas perguntas, com os dados que já estão no banco:

1. **Seca.** Os focos variam de ano para ano acompanhando condições mais secas? E, controlando por essas condições, a relação entre desmatamento e focos muda?
2. **Salto de 2017.** O aumento de focos em 2017 (e em 2018, no ano PRODES) parece fogo real, espalhado por muitos dias, ou concentrado em poucos dias, como seria um problema de processamento?

**Limite importante da pergunta 1:** o risco de fogo e os dias sem chuva vêm de cada foco detectado, ou seja, só existem onde e quando houve fogo. Eles são um indício de clima, não uma medida de seca no território. Um teste mais forte usaria uma série climática independente (por exemplo, o índice ENSO ou chuva observada), a ser acrescentada depois.

In [ ]:
import os
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine, text

load_dotenv(find_dotenv(usecwd=True))

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PASTA_FIGURAS = RAIZ / "figuras"
PASTA_FIGURAS.mkdir(exist_ok=True)
(RAIZ / "data" / "processed").mkdir(parents=True, exist_ok=True)

COR_DESMAT = "#0072B2"
COR_FOGO = "#D55E00"
FONTE = "Fontes: INPE/BDQueimadas (AQUA_M-T) e INPE/PRODES. Ano PRODES = agosto/(N-1) a julho/N."

plt.rcParams.update({
    "font.size": 13,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
})


def rodape(fig, texto=FONTE, tamanho=9):
    # rodapé com quebra de linha, para não ser cortado na borda da figura
    largura = int(fig.get_figwidth() * 11)
    fig.text(0.01, 0.01, textwrap.fill(texto, largura), fontsize=tamanho, color="gray", va="bottom")


def milhar(valor, _posicao=None):
    if valor >= 1:
        return f"{valor:,.0f}".replace(",", ".")
    return f"{valor:g}".replace(".", ",")

In [ ]:
engine = create_engine(os.environ["DATABASE_URL"])
FILTRO = "estado = 'PARÁ' AND ano_prodes BETWEEN 2009 AND 2025"

anual = pd.read_sql(text(
    "SELECT ano_prodes AS ano, COUNT(*) AS focos, "
    "COUNT(dia_sem_chuva) AS com_dsc, AVG(dia_sem_chuva) AS dsc_media, "
    "percentile_cont(0.5) WITHIN GROUP (ORDER BY dia_sem_chuva) AS dsc_mediana, "
    "COUNT(risco_fogo) AS com_risco, AVG(risco_fogo) AS risco_medio, AVG(frp) AS frp_medio "
    f"FROM stg.focos_queimada WHERE {FILTRO} GROUP BY ano_prodes ORDER BY ano_prodes"), engine)

mensal = pd.read_sql(text(
    f"SELECT ano_prodes AS ano, mes, COUNT(*) AS focos FROM stg.focos_queimada WHERE {FILTRO} "
    "GROUP BY ano_prodes, mes"), engine)

diario = pd.read_sql(text(
    "SELECT CAST(data_hora AS date) AS dia, COUNT(*) AS focos FROM stg.focos_queimada "
    "WHERE estado = 'PARÁ' GROUP BY 1 ORDER BY 1"), engine)

comp = pd.read_sql(text(
    "SELECT ano, area_desmatada_km2 FROM dw.vw_comparativo_para_ano ORDER BY ano"), engine)

for col in ("dsc_media", "dsc_mediana", "risco_medio", "frp_medio"):
    anual[col] = pd.to_numeric(anual[col], errors="coerce")
comp["area_desmatada_km2"] = pd.to_numeric(comp["area_desmatada_km2"], errors="coerce")
base = comp.merge(anual, on="ano")
base["cobertura"] = np.minimum(base["com_risco"], base["com_dsc"]) / base["focos"]


def amostra(col):
    # risco de fogo e dias sem chuva: só anos em que pelo menos 50% dos focos têm o dado
    d = base.dropna(subset=[col])
    if col in ("risco_medio", "dsc_media", "dsc_mediana"):
        d = d[d["cobertura"] >= 0.5]
    return d


print(base.shape, "| dias com focos:", len(diario))

## 1. Os indicadores de clima estão disponíveis em todos os anos?
Se uma parte dos focos não tem `dia_sem_chuva` ou `risco_fogo` (valor -999 na origem, que virou nulo), os indicadores daquele ano são menos confiáveis.

In [ ]:
qual = base[["ano", "focos"]].copy()
qual["% sem dias sem chuva"] = 100 * (1 - base["com_dsc"] / base["focos"])
qual["% sem risco de fogo"] = 100 * (1 - base["com_risco"] / base["focos"])
qual.round(1).set_index("ano")

## 2. Quando o fogo acontece
Meses na ordem do ano PRODES (agosto a julho). Mostra a estação seca e quais anos tiveram picos fora do padrão.

In [ ]:
ordem_meses = [8, 9, 10, 11, 12, 1, 2, 3, 4, 5, 6, 7]
nomes_meses = ["ago", "set", "out", "nov", "dez", "jan", "fev", "mar", "abr", "mai", "jun", "jul"]
matriz = (mensal.pivot(index="ano", columns="mes", values="focos")
                .reindex(columns=ordem_meses).fillna(0))

fig, ax = plt.subplots(figsize=(11, 8))
im = ax.imshow(matriz.to_numpy(dtype=float), cmap="Oranges", aspect="auto")
ax.set_xticks(range(12))
ax.set_xticklabels(nomes_meses)
ax.set_yticks(range(len(matriz)))
ax.set_yticklabels(matriz.index)
ax.set_title("Pará: focos por mês em cada ano PRODES", loc="left")
barra = fig.colorbar(im, ax=ax, shrink=0.8)
barra.set_label("Focos no mês")
barra.ax.yaxis.set_major_formatter(FuncFormatter(milhar))
for lado in ax.spines.values():
    lado.set_visible(False)

rodape(fig)
fig.tight_layout(rect=(0, 0.05, 1, 1))
fig.savefig(PASTA_FIGURAS / "09_mapa_calor_mensal.png", dpi=200)
plt.show()

## 3. Hipótese da seca
Cada ponto é um ano PRODES. Risco de fogo e dias sem chuva só entram nos anos em que pelo menos metade dos focos tem o dado (a tabela da seção 1 mostra quais). Se o fogo acompanhasse condições mais secas, esperaríamos relação positiva entre os focos e o risco de fogo (ou os dias sem chuva) dos próprios focos.

In [ ]:
indicadores = [
    ("risco_medio", "Risco de fogo médio dos focos"),
    ("dsc_mediana", "Dias sem chuva dos focos (mediana)"),
    ("frp_medio", "FRP médio dos focos (MW)"),
]

fig, eixos = plt.subplots(1, 3, figsize=(16, 5.8))
for ax, (col, rotulo) in zip(eixos, indicadores):
    d = amostra(col)
    rho = d[col].rank().corr(d["focos"].rank())
    ax.scatter(d[col], d["focos"], color=COR_FOGO, s=55, alpha=0.85, zorder=3)
    for _, r in d.iterrows():
        ax.annotate(str(int(r["ano"])), (r[col], r["focos"]),
                    textcoords="offset points", xytext=(4, 4), fontsize=8)
    ax.set_xlabel(rotulo)
    ax.set_title(f"Spearman = {rho:+.2f} (n = {len(d)})", fontsize=12, loc="left")
    ax.yaxis.set_major_formatter(FuncFormatter(milhar))
eixos[0].set_ylabel("Focos no ano PRODES")

rodape(fig)
fig.tight_layout(rect=(0, 0.06, 1, 1))
fig.savefig(PASTA_FIGURAS / "10_indicadores_seca.png", dpi=200)
plt.show()

### O desmatamento e os focos, controlando o indicador de clima
Correlação parcial de postos: a relação entre área desmatada e focos depois de retirar o que cada um tem em comum com o indicador. Se a relação negativa do estado vem de anos mais secos ou mais úmidos, ela deve mudar aqui.

In [ ]:
def residuo(y, x):
    coef = np.polyfit(x, y, 1)
    return y - np.polyval(coef, x)


def parcial(a, b, c):
    ra, rb, rc = a.rank().to_numpy(), b.rank().to_numpy(), c.rank().to_numpy()
    return np.corrcoef(residuo(ra, rc), residuo(rb, rc))[0, 1]


linhas = []
for col, rotulo in indicadores:
    d = amostra(col)
    linhas.append({
        "indicador": rotulo,
        "anos": f"{int(d['ano'].min())}-{int(d['ano'].max())}",
        "n": len(d),
        "focos x indicador": d["focos"].rank().corr(d[col].rank()),
        "área x indicador": d["area_desmatada_km2"].rank().corr(d[col].rank()),
        "área x focos (sem controle)": d["area_desmatada_km2"].rank().corr(d["focos"].rank()),
        "área x focos (controlando o indicador)": parcial(d["area_desmatada_km2"], d["focos"], d[col]),
    })

tabela = pd.DataFrame(linhas).round(2)
tabela.to_csv(RAIZ / "data" / "processed" / "testes_seca.csv", index=False)
tabela

## 4. O salto de 2017
Em ano civil, 2017 tem muito mais focos que 2016 e 2018. Se isso fosse problema de processamento, esperaríamos poucos dias com contagens anormais; se for fogo real, esperaríamos um aumento espalhado por muitos dias da estação seca.

In [ ]:
diario["dia"] = pd.to_datetime(diario["dia"])
diario["focos"] = pd.to_numeric(diario["focos"]).astype(int)
diario["ano"] = diario["dia"].dt.year

por_ano = diario.groupby("ano")["focos"].agg(total="sum", max_dia="max", dias_com_focos="count")
por_ano["top5_dias_pct"] = diario.groupby("ano")["focos"].apply(
    lambda s: 100 * s.nlargest(5).sum() / s.sum())
print("Os 10 dias com mais focos em toda a série:")
print(diario.nlargest(10, "focos")[["dia", "focos"]].to_string(index=False))
por_ano.round(1)

In [ ]:
anos_cmp = [2015, 2016, 2017, 2018, 2019]
cores = {2015: "#999999", 2016: COR_DESMAT, 2017: COR_FOGO, 2018: "#009E73", 2019: "#CC79A7"}

fig, ax = plt.subplots(figsize=(11, 6))
for ano in anos_cmp:
    serie = (diario[diario["ano"] == ano].set_index("dia")["focos"]
             .asfreq("D", fill_value=0).rolling(7, center=True, min_periods=1).mean())
    serie = serie[serie.index.month >= 7]
    x = [d.replace(year=2001) for d in serie.index]
    ax.plot(x, serie.to_numpy(), color=cores[ano], label=str(ano),
            linewidth=3 if ano == 2017 else 1.5, alpha=1 if ano == 2017 else 0.75)

ticks = [pd.Timestamp(2001, m, 1) for m in range(7, 13)]
ax.set_xticks(ticks)
ax.set_xticklabels(["jul", "ago", "set", "out", "nov", "dez"])
ax.set_ylabel("Focos por dia (média móvel de 7 dias)")
ax.yaxis.set_major_formatter(FuncFormatter(milhar))
ax.set_title("Pará: focos por dia na segunda metade do ano", loc="left")
ax.legend(frameon=False, ncol=5, loc="upper left")

rodape(fig)
fig.tight_layout(rect=(0, 0.05, 1, 1))
fig.savefig(PASTA_FIGURAS / "11_focos_diarios_2015_2019.png", dpi=200)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
cores_barras = [COR_FOGO if a == 2017 else COR_DESMAT for a in por_ano.index]
ax.bar(por_ano.index, por_ano["top5_dias_pct"], color=cores_barras)
ax.set_xticks(list(por_ano.index))
ax.tick_params(axis="x", rotation=45)
ax.set_ylabel("% dos focos do ano nos 5 dias mais intensos")
ax.set_title("Pará: quanto do ano se concentra em 5 dias", loc="left")

rodape(fig, "Fonte: INPE/BDQueimadas (AQUA_M-T). Anos civis, 2008–2026 (2026 até 13/10).")
fig.tight_layout(rect=(0, 0.05, 1, 1))
fig.savefig(PASTA_FIGURAS / "12_concentracao_em_5_dias.png", dpi=200)
plt.show()

## 5. Teste com o ENSO (ONI)
Definido antes de olhar os resultados: média mensal do ONI de agosto a novembro do **ano anterior** ao ano PRODES (a janela em que está o fogo no mapa de calor), direção esperada positiva (ONI mais alto, mais focos) e três coeficientes: focos x ONI, área x ONI e a relação área x focos controlando o ONI.

O CSV vem do Portal de Conhecimento sobre Mudanças Climáticas do Banco Mundial (página do ENSO). Salve em `data/raw/clima_portal_bm/oni_mensal.csv`.

In [ ]:
meses_en = {m: i for i, m in enumerate(
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], start=1)}
oni = pd.read_csv(RAIZ / "data" / "raw" / "clima_portal_bm" / "oni_mensal.csv")
oni["ano"] = oni["Date"].str[:4].astype(int)
oni["mes"] = oni["Date"].str[5:].map(meses_en)
janela = oni[oni["mes"].between(8, 11)].groupby("ano")["ONI value"].mean()

base["oni_ago_nov"] = base["ano"].map(lambda a: janela.get(a - 1, np.nan))
d = base.dropna(subset=["oni_ago_nov"])

enso = pd.DataFrame([{
    "n": len(d),
    "focos x ONI": d["focos"].rank().corr(d["oni_ago_nov"].rank()),
    "área x ONI": d["area_desmatada_km2"].rank().corr(d["oni_ago_nov"].rank()),
    "área x focos (sem controle)": d["area_desmatada_km2"].rank().corr(d["focos"].rank()),
    "área x focos (controlando o ONI)": parcial(d["area_desmatada_km2"], d["focos"], d["oni_ago_nov"]),
}]).round(2)
enso.to_csv(RAIZ / "data" / "processed" / "testes_enso.csv", index=False)
enso

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6.5))
ax.axvspan(-0.5, 0.5, color="#eeeeee", zorder=0)
ax.scatter(d["oni_ago_nov"], d["focos"], color=COR_FOGO, s=60, zorder=3)
for _, r in d.iterrows():
    ax.annotate(str(int(r["ano"])), (r["oni_ago_nov"], r["focos"]),
                textcoords="offset points", xytext=(5, 5), fontsize=9)
rho_enso = d["focos"].rank().corr(d["oni_ago_nov"].rank())
ax.set_xlabel("ONI médio de agosto a novembro do ano anterior (faixa cinza: neutro)")
ax.set_ylabel("Focos no ano PRODES")
ax.yaxis.set_major_formatter(FuncFormatter(milhar))
ax.set_title(f"Focos x ENSO: Spearman = {rho_enso:+.2f} (n = {len(d)})", loc="left")

rodape(fig, FONTE + " ONI: Portal de Conhecimento sobre Mudanças Climáticas (Banco Mundial).")
fig.tight_layout(rect=(0, 0.07, 1, 1))
fig.savefig(PASTA_FIGURAS / "13_focos_x_enso.png", dpi=200)
plt.show()

## 6. Teste com a chuva observada (ERA5)
Definido antes de olhar os resultados: chuva total de agosto a novembro do **ano anterior** ao ano PRODES, direção esperada negativa (menos chuva, mais focos), e três coeficientes: focos x chuva, área x chuva e a relação área x focos controlando a chuva.

O JSON vem do portal de dados do Banco Mundial (variável `pr`, produto série temporal, agregação mensal). Salve em `data/raw/clima_portal_bm/chuva_era5_para_mensal.json`. O arquivo traz só o código da unidade (por exemplo, `BRA.37678`); confira no portal que ele corresponde ao Pará.

In [ ]:
import json

caminho = RAIZ / "data" / "raw" / "clima_portal_bm" / "chuva_era5_para_mensal.json"
bruto = json.loads(caminho.read_text(encoding="utf-8"))
((codigo_area, serie),) = bruto["data"].items()
chuva = pd.Series(serie)
chuva.index = pd.PeriodIndex(chuva.index, freq="M")
cdf = chuva.rename("pr_mm").to_frame()
cdf["ano"] = cdf.index.year
cdf["mes"] = cdf.index.month
chuva_ago_nov = cdf[cdf["mes"].between(8, 11)].groupby("ano")["pr_mm"].sum()

base["chuva_ago_nov"] = base["ano"].map(lambda a: chuva_ago_nov.get(a - 1, np.nan))
d = base.dropna(subset=["chuva_ago_nov"])

teste_chuva = pd.DataFrame([{
    "unidade": codigo_area,
    "n": len(d),
    "focos x chuva": d["focos"].rank().corr(d["chuva_ago_nov"].rank()),
    "área x chuva": d["area_desmatada_km2"].rank().corr(d["chuva_ago_nov"].rank()),
    "área x focos (sem controle)": d["area_desmatada_km2"].rank().corr(d["focos"].rank()),
    "área x focos (controlando a chuva)": parcial(d["area_desmatada_km2"], d["focos"], d["chuva_ago_nov"]),
}]).round(2)
teste_chuva.to_csv(RAIZ / "data" / "processed" / "testes_chuva.csv", index=False)
teste_chuva

In [ ]:
mais_secos = d.nsmallest(3, "chuva_ago_nov")["ano"].tolist()

fig, ax = plt.subplots(figsize=(10, 6.5))
cores_pontos = [COR_FOGO if a in mais_secos else COR_DESMAT for a in d["ano"]]
ax.scatter(d["chuva_ago_nov"], d["focos"], c=cores_pontos, s=65, zorder=3)
for _, r in d.iterrows():
    ax.annotate(str(int(r["ano"])), (r["chuva_ago_nov"], r["focos"]),
                textcoords="offset points", xytext=(5, 5), fontsize=9)
rho_chuva = d["focos"].rank().corr(d["chuva_ago_nov"].rank())
ax.set_xlabel("Chuva total de agosto a novembro do ano anterior (mm)  |  laranja: os 3 períodos mais secos")
ax.set_ylabel("Focos no ano PRODES")
ax.yaxis.set_major_formatter(FuncFormatter(milhar))
ax.set_title(f"Focos x chuva da estação seca: Spearman = {rho_chuva:+.2f} (n = {len(d)})", loc="left")

rodape(fig, FONTE + " Chuva: ERA5, Portal de Conhecimento sobre Mudanças Climáticas (Banco Mundial).")
fig.tight_layout(rect=(0, 0.07, 1, 1))
fig.savefig(PASTA_FIGURAS / "14_focos_x_chuva.png", dpi=200)
plt.show()

## 7. Conferência com o CHIRPS
Mesmo teste da seção 6 (chuva de agosto a novembro do ano anterior, direção esperada negativa), com outra fonte de chuva: o CHIRPS (satélite e estações, grade de 0,05°), calculado pelo script `src/chuva_chirps.py`. Se as duas fontes concordam, o resultado não depende de uma só base de dados.

In [ ]:
chirps = pd.read_csv(RAIZ / "data" / "processed" / "chuva_chirps_mensal.csv")
estado = chirps[chirps["nivel"] == "estado"]
chirps_ago_nov = estado[estado["mes"].between(8, 11)].groupby("ano")["chuva_mm"].sum()

base["chuva_chirps"] = base["ano"].map(lambda a: chirps_ago_nov.get(a - 1, np.nan))
d2 = base.dropna(subset=["chuva_chirps", "chuva_ago_nov"])

tabela_chirps = pd.DataFrame([{
    "n": len(d2),
    "focos x chuva (CHIRPS)": d2["focos"].rank().corr(d2["chuva_chirps"].rank()),
    "focos x chuva (ERA5)": d2["focos"].rank().corr(d2["chuva_ago_nov"].rank()),
    "área x chuva (CHIRPS)": d2["area_desmatada_km2"].rank().corr(d2["chuva_chirps"].rank()),
    "área x focos (controlando CHIRPS)": parcial(d2["area_desmatada_km2"], d2["focos"], d2["chuva_chirps"]),
    "concordância ERA5 x CHIRPS": d2["chuva_ago_nov"].rank().corr(d2["chuva_chirps"].rank()),
}]).round(2)
tabela_chirps.to_csv(RAIZ / "data" / "processed" / "testes_chuva_chirps.csv", index=False)
tabela_chirps

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
# deslocamentos para não sobrepor rótulos de anos vizinhos
AJ_ESQ = {2024: (-34, -6), 2015: (-34, 4)}
AJ_DIR = {2021: (-38, 6), 2020: (-32, -13), 2009: (6, -11)}

rho_ch = d2["focos"].rank().corr(d2["chuva_chirps"].rank())
ax1.scatter(d2["chuva_chirps"], d2["focos"], color=COR_DESMAT, s=60, zorder=3)
for _, r in d2.iterrows():
    ax1.annotate(str(int(r["ano"])), (r["chuva_chirps"], r["focos"]),
                 textcoords="offset points", xytext=AJ_ESQ.get(int(r["ano"]), (5, 5)), fontsize=8)
ax1.set_xlabel("Chuva CHIRPS, ago–nov do ano anterior (mm)")
ax1.set_ylabel("Focos no ano PRODES")
ax1.yaxis.set_major_formatter(FuncFormatter(milhar))
ax1.set_title(f"CHIRPS: Spearman = {rho_ch:+.2f} (n = {len(d2)})", loc="left", fontsize=13)

conc = d2["chuva_ago_nov"].rank().corr(d2["chuva_chirps"].rank())
ax2.scatter(d2["chuva_ago_nov"], d2["chuva_chirps"], color=COR_FOGO, s=60, zorder=3)
for _, r in d2.iterrows():
    ax2.annotate(str(int(r["ano"])), (r["chuva_ago_nov"], r["chuva_chirps"]),
                 textcoords="offset points", xytext=AJ_DIR.get(int(r["ano"]), (5, 5)), fontsize=8)
ax2.set_xlabel("Chuva ERA5, ago–nov do ano anterior (mm)")
ax2.set_ylabel("Chuva CHIRPS (mm)")
ax2.set_title(f"Concordância das fontes: Spearman = {conc:+.2f}", loc="left", fontsize=13)

rodape(fig, FONTE + " Chuva: CHIRPS (UCSB/CHC) e ERA5 (Banco Mundial).")
fig.tight_layout(rect=(0, 0.07, 1, 1))
fig.savefig(PASTA_FIGURAS / "15_chuva_chirps_x_era5.png", dpi=200)
plt.show()

**Como ler estes resultados**
- Os indicadores de clima só existem onde houve foco: servem de indício, não de medida de seca no território.
- Com 17 anos, os coeficientes descrevem estes dados e não provam causa. A correlação parcial com n = 17 é bastante incerta.
- Um dia com muitos focos pode ser fogo real (uma onda de calor, por exemplo) ou problema de detecção; este notebook só diz se o padrão de 2017 destoa dos outros anos, não por quê.
- Esta análise não testa a hipótese de fogo em áreas já desmatadas, que exige cruzar cada foco com os polígonos do PRODES.